# Adding transformations to multiscale images

Adding transformations to individual multiscale images is a way to enrich the metadata associated with the image.
THis can be useful to inform implementations on how to view or interpret the image data,
i.e. if an image is supposed to be viewed with a rotation or a shear applied.

In [2]:
from ome_zarr import OMEZarrImage, OMEZarrMultiscale
import numpy as np

In [5]:
data = np.random.rand(256, 256)
image = OMEZarrImage(data=data, axes="yx", name="My Image")

ms = OMEZarrMultiscale(image=image)

We can inspect the metadata of the multiscale image to see what coordinate systems are currently present:

In [6]:
ms.metadata.coordinateSystems

(CoordinateSystem(name='physical', axes=(Axis(name='y', type='space', discrete=None, unit=None, longName=None), Axis(name='x', type='space', discrete=None, unit=None, longName=None))),)

To attach another transformation to the multiscale image, we need to specify both the transform as well as the coordinate system it outputs to.
If we assume that we specify a rotation transformation that outputs into a "world" coordinate system,
we can create this structure as follows:

In [8]:
rotation_transform = {
    "type": "rotation",
    "rotation": [
        [0, -1],
        [1, 0]
    ],
    "input": {"name": "physical"},
    "output": {"name": "world"}
}

world_cs = {
    "name": "world",
    "axes": [
        {"name": "x", "type": "space"},
        {"name": "y", "type": "space"}
    ],
}

The `output` field refers to the name `world` of the created coordinate system.
The input field refers to the intrinsic/physical coordinate system of the image, which defaults to `physical` if not specified otherwise.
We can then just pass these to the `OMEZarrMultiscale` constructor as follows:

In [9]:
ms = OMEZarrMultiscale(
    image=image,
    coordinateTransformations=[rotation_transform],
    coordinate_systems=[world_cs],
    )

And that's it! A viewer could now choose to show your image in the "world" coordinate system,
which would apply the rotation transformation to the image data.

For more information of how to construct transformations, see the [respective section of the specification](https://ngff.openmicroscopy.org/specifications/dev/index.html#coordinatetransformations-metadata).